# 04 · Neighborhood clustering: which zones won, lost or stayed the same?

For every Uber pickup zone, measure how five KPIs changed from **Feb–Dec 2024** to **Feb–Dec 2025** (same months, so no seasonality; January skipped because the toll started Jan 5). Then **K-means** groups zones with similar change patterns. The toll exposure of each zone (share of its trips charged the toll) is **not** used for clustering, only to interpret the clusters afterward.

In [ ]:
import os, sys
os.chdir(r"C:\Users\lalit\Documents\nyc-congestion-toll-impact")
sys.path.append("src")

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from zones import CBD_ZONES, BOUNDARY_ZONES

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

## 1. KPI changes per pickup zone

In [ ]:
agg = duckdb.sql("""
    SELECT pu_zone,
           CASE WHEN trip_date < DATE '2025-01-01' THEN 'b' ELSE 'a' END AS period,
           SUM(trips) AS trips,
           SUM(sum_rider_cost) AS cost,
           SUM(sum_miles) AS miles,
           SUM(sum_trip_seconds) AS secs,
           SUM(sum_driver_pay) AS pay,
           SUM(sum_wait_seconds) AS wait_s,
           SUM(wait_trips) AS wait_n,
           SUM(tolled_trips) AS tolled
    FROM 'data/summary/*.parquet'
    WHERE company = 'Uber'
      AND ((trip_date BETWEEN '2024-02-01' AND '2024-12-31')
        OR (trip_date BETWEEN '2025-02-01' AND '2025-12-31'))
    GROUP BY ALL
""").df()

w = agg.pivot(index="pu_zone", columns="period")
w.columns = [f"{m}_{p}" for m, p in w.columns]
w = w.dropna()

z = pd.DataFrame(index=w.index)
z["trips_2024"] = w["trips_b"]
z["trips_pct"] = (w["trips_a"] / w["trips_b"] - 1) * 100
z["cost_pct"] = ((w["cost_a"] / w["trips_a"]) / (w["cost_b"] / w["trips_b"]) - 1) * 100
z["speed_pct"] = ((w["miles_a"] / w["secs_a"]) / (w["miles_b"] / w["secs_b"]) - 1) * 100
z["pay_hr_pct"] = ((w["pay_a"] / w["secs_a"]) / (w["pay_b"] / w["secs_b"]) - 1) * 100
z["wait_change_min"] = (w["wait_s_a"] / w["wait_n_a"] - w["wait_s_b"] / w["wait_n_b"]) / 60
z["toll_exposure_pct"] = w["tolled_a"] / w["trips_a"] * 100      # for interpretation only

# keep zones with enough volume for stable percentages
z = z[z["trips_2024"] >= 20000]

lookup = pd.read_csv("data/taxi_zone_lookup.csv").set_index("LocationID")
z = z.join(lookup[["Borough", "Zone"]])
z["area"] = np.where(z.index.isin(CBD_ZONES), "Toll zone",
             np.where(z.index.isin(BOUNDARY_ZONES), "Boundary", z["Borough"]))

print(len(z), "zones kept")
citywide = z[["trips_pct", "cost_pct", "speed_pct", "pay_hr_pct", "wait_change_min"]].median()
print("\nCitywide median change:\n", citywide.round(2))

## 2. Choose the number of clusters

In [ ]:
FEATURES = ["trips_pct", "cost_pct", "speed_pct", "pay_hr_pct", "wait_change_min"]

X = z[FEATURES].copy()
X = X.clip(X.quantile(0.01), X.quantile(0.99), axis=1)     # soften extreme zones
Xs = StandardScaler().fit_transform(X)

scores = {}
for k in range(3, 8):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(Xs)
    scores[k] = silhouette_score(Xs, labels)
pd.Series(scores, name="silhouette").round(3)

## 3. Fit K-means and profile the clusters

K defaults to the best silhouette score. You can override it (e.g. `K = 4`) if a nearby value gives clusters that are easier to explain.

In [ ]:
K = max(scores, key=scores.get)
km = KMeans(n_clusters=K, n_init=20, random_state=42).fit(Xs)
z["cluster"] = km.labels_

profile = z.groupby("cluster").agg(
    zones=("trips_pct", "size"),
    trips_2024_M=("trips_2024", lambda s: s.sum() / 1e6),
    **{f: (f, "median") for f in FEATURES},
    toll_exposure_pct=("toll_exposure_pct", "median"),
).round(2)
print("K =", K)
profile

In [ ]:
# where the clusters are: share of each cluster's zones by area
pd.crosstab(z["cluster"], z["area"])

In [ ]:
# the 5 busiest zones in each cluster
for k, g in z.sort_values("trips_2024", ascending=False).groupby("cluster"):
    print(f"\nCluster {k}:", ", ".join(g["Zone"].head(5)))

## 4. Chart: trips change vs speed change

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.tab10.colors
for k, g in z.groupby("cluster"):
    ax.scatter(g["speed_pct"], g["trips_pct"], s=g["trips_2024"] / 20000, alpha=0.7,
               color=colors[k], label=f"Cluster {k} ({len(g)} zones)", edgecolor="white", lw=0.5)
ax.axhline(citywide["trips_pct"], color="grey", ls="--", lw=0.8)
ax.axvline(citywide["speed_pct"], color="grey", ls="--", lw=0.8)
ax.set_xlabel("Speed change, 2025 vs 2024 (%)")
ax.set_ylabel("Trips change, 2025 vs 2024 (%)")
ax.set_title("Uber pickup zones: change after the toll (bubble size = 2024 trips)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/zone_clusters_uber.png", dpi=150)
plt.show()

In [ ]:
# export for the Tableau map later
out = z.reset_index().rename(columns={"pu_zone": "LocationID"})
out.round(3).to_csv("outputs/zone_clusters_uber.csv", index=False)
print("Saved", len(out), "zones")